<a href="https://colab.research.google.com/github/sr606/LLM/blob/main/mermaid_v7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
🔵 1️⃣ main.py
# main.py
from agent import run_agent

if __name__ == "__main__":
    run_agent()
🔵 2️⃣ agent.py (FULL CLEAN VERSION)
import os
import json

from parser import split_into_stages
from deterministic_parser import parse_stage_technical
from llm_service import LLMService

from graph_builder import build_graph
from pipeline_detector import detect_pipelines
from pipeline_name import generate_pipeline_name

from renderer import render_detailed_pipeline
from high_level_renderer import render_high_level_architecture
from technical_pdf_writer import generate_technical_pdf


INPUT_FOLDER = "../data/input"
OUTPUT_FOLDER = "../data/output"


def run_agent():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    llm = LLMService()

    for file in os.listdir(INPUT_FOLDER):
        if not file.endswith(".txt"):
            continue

        input_path = os.path.join(INPUT_FOLDER, file)

        with open(input_path, "r", encoding="utf-8") as f:
            pseudocode = f.read()

        print(f"\nProcessing: {file}")

        # ---------------------------
        # Split stages
        # ---------------------------
        stages = split_into_stages(pseudocode)

        stage_metadata = []
        technical_metadata = []

        # ---------------------------
        # Process each stage
        # ---------------------------
        for stage in stages:
            stage_id = stage["stage_id"]
            block = stage["block"]

            # Deterministic technical parsing
            tech = parse_stage_technical(stage_id, block)
            technical_metadata.append(tech)

            # LLM only for summary
            summary = llm.generate_summary(block)

            stage_metadata.append({
                "stage_name": stage_id,
                "stage_type": tech["stage_type"],
                "summary": summary
            })

        # ---------------------------
        # Save metadata
        # ---------------------------
        base_name = file.replace(".txt", "")
        metadata_path = os.path.join(OUTPUT_FOLDER, f"{base_name}_metadata.json")

        with open(metadata_path, "w", encoding="utf-8") as f:
            json.dump(technical_metadata, f, indent=4)

        # ---------------------------
        # Build stage graph
        # ---------------------------
        stage_id_order = [s["stage_id"] for s in stages]
        graph = build_graph(pseudocode, stage_metadata, stage_id_order)

        pipelines = detect_pipelines(graph, include_singletons=False)

        pipeline_names = [
            generate_pipeline_name(graph, p)
            for p in pipelines
        ]

        # ---------------------------
        # Render Detailed Stage Diagram
        # ---------------------------
        render_detailed_pipeline(
            graph,
            pipelines,
            pipeline_names,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_detailed")
        )

        # ---------------------------
        # Render High Level Architecture
        # ---------------------------
        render_high_level_architecture(
            technical_metadata,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_high_level")
        )

        # ---------------------------
        # Generate Technical PDF
        # ---------------------------
        generate_technical_pdf(
            technical_metadata,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_technical.pdf")
        )

        print(f"Completed: {file}")
🔵 3️⃣ deterministic_parser.py
import re


def parse_stage_technical(stage_id: str, stage_block: str):
    result = {
        "stage_id": stage_id,
        "stage_type": extract_stage_type(stage_block),
        "input_datasets": [],
        "output_datasets": [],
        "joins": [],
        "filters": [],
        "aggregations": [],
        "column_mappings": []
    }

    # Inputs
    result["input_datasets"] = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)",
        stage_block
    )

    # Outputs
    result["output_datasets"] = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)",
        stage_block
    )

    # Joins
    join_pattern = r"(INNER|LEFT|RIGHT|FULL|LEFT OUTER|RIGHT OUTER)?\s*JOIN\s+([\w\.]+)\s+ON\s+(.*?)(?:\n|WHERE|GROUP BY|ORDER BY)"
    joins = re.findall(join_pattern, stage_block, re.IGNORECASE | re.DOTALL)

    for jt, table, condition in joins:
        result["joins"].append({
            "join_type": (jt or "JOIN").upper(),
            "table": table.strip(),
            "condition": condition.strip()
        })

    # Filters
    where_pattern = r"WHERE\s+(.*?)(?:GROUP BY|ORDER BY|$)"
    filters = re.findall(where_pattern, stage_block, re.IGNORECASE | re.DOTALL)
    result["filters"] = [f.strip() for f in filters]

    # Aggregations
    agg_pattern = r"(SUM|COUNT|AVG|MIN|MAX)\((.*?)\)"
    aggs = re.findall(agg_pattern, stage_block, re.IGNORECASE)
    result["aggregations"] = [f"{f.upper()}({c})" for f, c in aggs]

    # Column mappings (Transformer stages)
    mapping_pattern = r"(\w+)\s*=\s*([A-Za-z0-9_\.]+)"
    mappings = re.findall(mapping_pattern, stage_block)

    for target, source in mappings:
        result["column_mappings"].append({
            "source": source,
            "target": target
        })

    return result


def extract_stage_type(block):
    match = re.search(r"StageType:\s*(\w+)", block)
    return match.group(1) if match else "Unknown"
🔵 4️⃣ llm_service.py (SUMMARY ONLY)
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()


class LLMService:

    def __init__(self):
        self.client = AzureOpenAI(
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        )
        self.deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

    def generate_summary(self, stage_block: str):

        prompt = """
Summarize the following ETL stage in 3 concise bullet points.
Do not infer anything not present.
Be technical and precise.
Return only bullet points.
"""

        response = self.client.chat.completions.create(
            model=self.deployment,
            temperature=0.0,
            messages=[
                {"role": "system", "content": "You summarize ETL stages."},
                {"role": "user", "content": prompt + stage_block}
            ],
        )

        text = response.choices[0].message.content.strip()
        return [line.strip("- ").strip() for line in text.split("\n") if line.strip()]
🔵 5️⃣ renderer.py (Detailed Stage Diagram)
from graphviz import Digraph


def render_detailed_pipeline(graph, pipelines, pipeline_names, output_path):
    dot = Digraph("Detailed_Pipeline", format="pdf")
    dot.attr(rankdir="LR")

    dot.node_attr.update(
        shape="box",
        style="rounded,filled",
        fillcolor="#ECEFF1",
        fontname="Helvetica",
        fontsize="10"
    )

    for idx, pipeline in enumerate(pipelines):
        with dot.subgraph(name=f"cluster_{idx}") as sub:
            sub.attr(label=pipeline_names[idx], style="rounded")

            for stage_id in pipeline:
                node = graph.nodes[stage_id]
                label = f"{node.display_name}\n({node.stage_type})"
                sub.node(stage_id, label)

    for src, tgt in graph.edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)
🔵 6️⃣ high_level_renderer.py
from graphviz import Digraph


def render_high_level_architecture(technical_metadata, output_path):
    dot = Digraph("High_Level", format="pdf")
    dot.attr(rankdir="LR")

    dot.node("Sources", shape="box", style="filled", fillcolor="#BBDEFB")
    dot.node("Transformations", shape="box", style="filled", fillcolor="#FFE0B2")
    dot.node("Targets", shape="box", style="filled", fillcolor="#C8E6C9")

    dot.edge("Sources", "Transformations")
    dot.edge("Transformations", "Targets")

    dot.render(output_path, cleanup=True)
🔵 7️⃣ technical_pdf_writer.py
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, ListFlowable, ListItem
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet


def generate_technical_pdf(technical_metadata, output_path):

    doc = SimpleDocTemplate(output_path)
    elements = []

    styles = getSampleStyleSheet()
    normal = styles["Normal"]

    for stage in technical_metadata:

        elements.append(Paragraph(f"<b>Stage:</b> {stage['stage_id']}", styles["Heading3"]))
        elements.append(Spacer(1, 10))

        bullets = []

        if stage["input_datasets"]:
            bullets.append(f"Inputs: {', '.join(stage['input_datasets'])}")

        if stage["output_datasets"]:
            bullets.append(f"Outputs: {', '.join(stage['output_datasets'])}")

        for j in stage["joins"]:
            bullets.append(f"{j['join_type']} JOIN {j['table']} ON {j['condition']}")

        for f in stage["filters"]:
            bullets.append(f"Filter: {f}")

        for a in stage["aggregations"]:
            bullets.append(f"Aggregation: {a}")

        for m in stage["column_mappings"][:10]:
            bullets.append(f"{m['source']} → {m['target']}")

        flow = ListFlowable(
            [ListItem(Paragraph(b, normal)) for b in bullets],
            bulletType='bullet'
        )

        elements.append(flow)
        elements.append(Spacer(1, 20))

    doc.build(elements)

In [ ]:
🔵 1️⃣ main.py
from agent import run_agent

if __name__ == "__main__":
    run_agent()
🔵 2️⃣ agent.py
import os

from parser import split_into_stages
from deterministic_parser import parse_stage_technical
from graph_builder import build_graph
from renderer import render_detailed_pipeline
from high_level_renderer import render_high_level_architecture


INPUT_FOLDER = "data/input"
OUTPUT_FOLDER = "data/output"


def run_agent():

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    for file in os.listdir(INPUT_FOLDER):
        if not file.endswith(".txt"):
            continue

        input_path = os.path.join(INPUT_FOLDER, file)

        with open(input_path, "r", encoding="utf-8") as f:
            pseudocode = f.read()

        print(f"\nProcessing: {file}")

        stages = split_into_stages(pseudocode)

        technical_metadata = []

        for stage in stages:
            tech = parse_stage_technical(stage["stage_id"], stage["block"])
            technical_metadata.append(tech)

        # Build stage graph
        graph = build_graph(technical_metadata)

        base_name = file.replace(".txt", "")

        # Detailed Diagram (column-level but clean)
        render_detailed_pipeline(
            graph,
            technical_metadata,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_detailed")
        )

        # High Level Diagram
        render_high_level_architecture(
            technical_metadata,
            os.path.join(OUTPUT_FOLDER, f"{base_name}_high_level")
        )

        print(f"Completed: {file}")
🔵 3️⃣ parser.py
import re

def split_into_stages(text: str):

    pattern = r"// --- \[(.*?)\] ---([\s\S]*?)(?=// --- \[|$)"
    matches = re.findall(pattern, text)

    stages = []

    for header, body in matches:
        parts = header.split(":")
        if len(parts) < 2:
            continue

        stage_id = parts[1].strip()
        stage_block = header + "\n" + body.strip()

        stages.append({
            "stage_id": stage_id,
            "block": stage_block
        })

    return stages
🔵 4️⃣ deterministic_parser.py

⚡ Important:
We extract ONLY:

input datasets

output datasets

transformation column mappings

ignore SQL select clutter

ignore unused columns

import re


def parse_stage_technical(stage_id, stage_block):

    result = {
        "stage_id": stage_id,
        "stage_type": extract_stage_type(stage_block),
        "input_datasets": [],
        "output_datasets": [],
        "transform_columns": []
    }

    # ---------------------------
    # Inputs
    # ---------------------------
    result["input_datasets"] = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)",
        stage_block
    )

    # ---------------------------
    # Outputs
    # ---------------------------
    result["output_datasets"] = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)",
        stage_block
    )

    # ---------------------------
    # Extract ONLY transformation mappings
    # Format: TARGET = SOURCE
    # ---------------------------
    mapping_pattern = r"^\s*(\w+)\s*=\s*([A-Za-z0-9_\.]+)"

    lines = stage_block.split("\n")

    for line in lines:
        match = re.match(mapping_pattern, line.strip())
        if match:
            target = match.group(1)
            source = match.group(2)

            result["transform_columns"].append({
                "target": target,
                "source": source
            })

    return result


def extract_stage_type(block):
    match = re.search(r"StageType:\s*(\w+)", block)
    return match.group(1) if match else "Unknown"

This now:

Ignores SQL huge block

Only captures transformation logic

Clean and deterministic

🔵 5️⃣ graph_model.py
class Node:
    def __init__(self, name, stage_type):
        self.name = name
        self.stage_type = stage_type


class Graph:
    def __init__(self):
        self.nodes = {}
        self.edges = []

    def add_node(self, node):
        self.nodes[node.name] = node

    def add_edge(self, source, target):
        if source != target:
            self.edges.append((source, target))
🔵 6️⃣ graph_builder.py
from graph_model import Graph, Node


def build_graph(technical_metadata):

    graph = Graph()

    dataset_producer = {}
    dataset_consumers = {}

    for stage in technical_metadata:

        stage_id = stage["stage_id"]

        graph.add_node(Node(stage_id, stage["stage_type"]))

        for inp in stage["input_datasets"]:
            dataset_consumers.setdefault(inp, []).append(stage_id)

        for out in stage["output_datasets"]:
            dataset_producer[out] = stage_id

    for dataset, producer in dataset_producer.items():
        consumers = dataset_consumers.get(dataset, [])

        for consumer in consumers:
            graph.add_edge(producer, consumer)

    return graph
🔵 7️⃣ renderer.py (Detailed Diagram)

Now shows:

Stage name

Stage type

Only transformation columns

from graphviz import Digraph


def render_detailed_pipeline(graph, technical_metadata, output_path):

    dot = Digraph("Detailed", format="pdf")
    dot.attr(rankdir="LR")

    dot.node_attr.update(
        shape="box",
        style="rounded,filled",
        fillcolor="#ECEFF1",
        fontname="Helvetica",
        fontsize="9"
    )

    # Create mapping for quick lookup
    metadata_map = {
        m["stage_id"]: m for m in technical_metadata
    }

    for stage_id, node in graph.nodes.items():

        meta = metadata_map.get(stage_id)

        label_lines = [
            f"{stage_id}",
            f"({node.stage_type})"
        ]

        # Only show transformation columns
        if meta["transform_columns"]:
            label_lines.append("---- Transform Columns ----")

            for col in meta["transform_columns"][:10]:
                label_lines.append(f"{col['source']} → {col['target']}")

        label = "\n".join(label_lines)

        dot.node(stage_id, label)

    for src, tgt in graph.edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)
🔵 8️⃣ high_level_renderer.py

This builds real layer classification:

If no inputs → Source

If no outputs → Target

Else → Transformation

from graphviz import Digraph


def render_high_level_architecture(technical_metadata, output_path):

    dot = Digraph("HighLevel", format="pdf")
    dot.attr(rankdir="LR")

    sources = []
    transforms = []
    targets = []

    for stage in technical_metadata:

        if not stage["input_datasets"]:
            sources.append(stage["stage_id"])

        elif not stage["output_datasets"]:
            targets.append(stage["stage_id"])

        else:
            transforms.append(stage["stage_id"])

    dot.node("Sources", "\n".join(sources) or "None",
             shape="box", style="filled", fillcolor="#BBDEFB")

    dot.node("Transformations", "\n".join(transforms) or "None",
             shape="box", style="filled", fillcolor="#FFE0B2")

    dot.node("Targets", "\n".join(targets) or "None",
             shape="box", style="filled", fillcolor="#C8E6C9")

    dot.edge("Sources", "Transformations")
    dot.edge("Transformations", "Targets")

    dot.render(output_path, cleanup=True)
🔵 9️⃣ requirements.txt
graphviz
🚀 How To Run

Inside project folder:

python -m venv venv
venv\Scripts\activate
pip install -r requirements.txt
python main.py
